In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # suppress verbose TensorFlow startup logs

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
tf.get_logger().setLevel("ERROR")

import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ipywidgets import interact, FloatSlider

from sklearn.datasets import load_breast_cancer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

tf.keras.utils.set_random_seed(42)
np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")
%matplotlib inline

print("Setup complete. You can start the experiment.")

ModuleNotFoundError: No module named 'numpy'

In [ ]:
# Load the Breast Cancer dataset

cancer = load_breast_cancer(as_frame=True)
cancer_df = cancer.frame
cancer_df

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.16220,0.66560,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.12380,0.18660,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.14440,0.42450,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.20980,0.86630,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.13740,0.20500,0.4000,0.1625,0.2364,0.07678,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,0.1726,0.05623,...,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115,0
565,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,0.1752,0.05533,...,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637,0
566,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,0.1590,0.05648,...,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820,0
567,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,0.2397,0.07016,...,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400,0


In [ ]:
X = cancer.data
y = cancer.target

print("Shape:", X.shape)
print("Class counts:\n", y.value_counts().sort_index())
print("Target names:", cancer.target_names)

Shape: (569, 30)
Class counts:
 target
0    212
1    357
Name: count, dtype: int64
Target names: ['malignant' 'benign']


In [ ]:
# Train / Validation / Test Split, then Standardize

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.2,
    random_state=42,
    stratify=y_train_full,
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Train shape:", X_train_scaled.shape)
print("Validation shape:", X_val_scaled.shape)
print("Test shape:", X_test_scaled.shape)

Train shape: (364, 30)
Validation shape: (91, 30)
Test shape: (114, 30)


In [ ]:
# Build the Perceptron Model

perceptron_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_scaled.shape[1],)),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])

perceptron_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

perceptron_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_4 (Dense)                 │ (None, 1)              │            31 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 31 (124.00 B)

 Trainable params: 31 (124.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Train the Perceptron with Early Stopping

perceptron_history = perceptron_model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=40,
    batch_size=32,
    verbose=0,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)],
)

print(f"Perceptron training finished after {len(perceptron_history.history['loss'])} epochs.")

Perceptron training finished after 40 epochs.


In [ ]:
# Evaluate the Perceptron on the Test Set

perceptron_test_probs = perceptron_model.predict(X_test_scaled, verbose=0).ravel()
perceptron_test_pred = (perceptron_test_probs >= 0.5).astype(int)

perceptron_accuracy = accuracy_score(y_test, perceptron_test_pred)
print(f"Perceptron Test Accuracy : {perceptron_accuracy:.4f}")
print(classification_report(y_test, perceptron_test_pred, target_names=cancer.target_names))

Perceptron Test Accuracy : 0.9737
              precision    recall  f1-score   support

   malignant       0.95      0.98      0.96        42
      benign       0.99      0.97      0.98        72

    accuracy                           0.97       114
   macro avg       0.97      0.97      0.97       114
weighted avg       0.97      0.97      0.97       114



In [ ]:
# Build the MLP Model

mlp_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_scaled.shape[1],)),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])

mlp_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

mlp_model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_5 (Dense)                 │ (None, 32)             │           992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,537 (6.00 KB)

 Trainable params: 1,537 (6.00 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Train the MLP with Early Stopping

mlp_history = mlp_model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=60,
    batch_size=32,
    verbose=0,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=7, restore_best_weights=True)],
)

print(f"MLP training finished after {len(mlp_history.history['loss'])} epochs.")

MLP training finished after 60 epochs.


In [ ]:
# Evaluate the MLP on the Test Set

mlp_test_probs = mlp_model.predict(X_test_scaled, verbose=0).ravel()
mlp_test_pred = (mlp_test_probs >= 0.5).astype(int)

mlp_accuracy = accuracy_score(y_test, mlp_test_pred)
print(f"MLP Test Accuracy : {mlp_accuracy:.4f}")
print(classification_report(y_test, mlp_test_pred, target_names=cancer.target_names))

MLP Test Accuracy : 0.9474
              precision    recall  f1-score   support

   malignant       0.91      0.95      0.93        42
      benign       0.97      0.94      0.96        72

    accuracy                           0.95       114
   macro avg       0.94      0.95      0.94       114
weighted avg       0.95      0.95      0.95       114



In [ ]:
# Compare Both Models on Test Metrics

comparison = []
for name, y_pred, y_prob in [
    ("Perceptron", perceptron_test_pred, perceptron_test_probs),
    ("MLP", mlp_test_pred, mlp_test_probs),
]:
    comparison.append([
        name,
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        f1_score(y_test, y_pred),
        roc_auc_score(y_test, y_prob),
    ])

comparison_df = pd.DataFrame(
    comparison,
    columns=["Model", "Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]
).round(4)

comparison_df

,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Perceptron,0.9737,0.9859,0.9722,0.9790,0.9960
1,MLP,0.9474,0.9714,0.9444,0.9577,0.9927


In [2]:
# Interactive Learning Curves: Accuracy (hover for exact values)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Perceptron Accuracy", "MLP Accuracy"))

fig.add_trace(go.Scatter(y=perceptron_history.history["accuracy"], mode="lines", name="Train",
                          line=dict(color="#2a9d8f")), row=1, col=1)
fig.add_trace(go.Scatter(y=perceptron_history.history["val_accuracy"], mode="lines", name="Validation",
                          line=dict(color="#e76f51")), row=1, col=1)

fig.add_trace(go.Scatter(y=mlp_history.history["accuracy"], mode="lines", name="Train",
                          line=dict(color="#2a9d8f"), showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(y=mlp_history.history["val_accuracy"], mode="lines", name="Validation",
                          line=dict(color="#e76f51"), showlegend=False), row=1, col=2)

fig.update_xaxes(title_text="Epoch")
fig.update_yaxes(title_text="Accuracy")
fig.update_layout(title="Training vs Validation Accuracy", height=450, width=950,
                   template="plotly_white")

fig.show()

NameError: name 'make_subplots' is not defined

In [ ]:
# Interactive Learning Curves: Loss (hover for exact values)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Perceptron Loss", "MLP Loss"))

fig.add_trace(go.Scatter(y=perceptron_history.history["loss"], mode="lines", name="Train",
                          line=dict(color="#2a9d8f")), row=1, col=1)
fig.add_trace(go.Scatter(y=perceptron_history.history["val_loss"], mode="lines", name="Validation",
                          line=dict(color="#e76f51")), row=1, col=1)

fig.add_trace(go.Scatter(y=mlp_history.history["loss"], mode="lines", name="Train",
                          line=dict(color="#2a9d8f"), showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(y=mlp_history.history["val_loss"], mode="lines", name="Validation",
                          line=dict(color="#e76f51"), showlegend=False), row=1, col=2)

fig.update_xaxes(title_text="Epoch")
fig.update_yaxes(title_text="Binary Cross-Entropy Loss")
fig.update_layout(title="Training vs Validation Loss", height=450, width=950,
                   template="plotly_white")

fig.show()

In [ ]:
# Interactive Confusion Matrices (hover for exact counts)

perceptron_cm = confusion_matrix(y_test, perceptron_test_pred)
mlp_cm = confusion_matrix(y_test, mlp_test_pred)
labels = list(cancer.target_names)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Perceptron", "MLP"))

fig.add_trace(go.Heatmap(z=perceptron_cm, x=labels, y=labels, colorscale="Blues",
                          text=perceptron_cm, texttemplate="%{text}", showscale=False), row=1, col=1)
fig.add_trace(go.Heatmap(z=mlp_cm, x=labels, y=labels, colorscale="Blues",
                          text=mlp_cm, texttemplate="%{text}", showscale=False), row=1, col=2)

fig.update_xaxes(title_text="Predicted Label")
fig.update_yaxes(title_text="True Label", autorange="reversed")
fig.update_layout(title="Confusion Matrix Comparison", height=450, width=950,
                   template="plotly_white")

fig.show()

In [ ]:
# Interactive ROC Curve Comparison (hover for threshold, TPR, FPR)

fpr_p, tpr_p, thr_p = roc_curve(y_test, perceptron_test_probs)
fpr_m, tpr_m, thr_m = roc_curve(y_test, mlp_test_probs)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=fpr_p, y=tpr_p, mode="lines", name=f"Perceptron (AUC = {roc_auc_score(y_test, perceptron_test_probs):.3f})",
    line=dict(color="#2a9d8f"),
    hovertemplate="FPR: %{x:.3f}<br>TPR: %{y:.3f}<extra></extra>"
))
fig.add_trace(go.Scatter(
    x=fpr_m, y=tpr_m, mode="lines", name=f"MLP (AUC = {roc_auc_score(y_test, mlp_test_probs):.3f})",
    line=dict(color="#e76f51"),
    hovertemplate="FPR: %{x:.3f}<br>TPR: %{y:.3f}<extra></extra>"
))
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode="lines", name="Random Guess",
    line=dict(color="gray", dash="dash")
))

fig.update_layout(
    title="ROC Curve Comparison",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    height=500, width=650,
    template="plotly_white"
)

fig.show()

In [ ]:
# Interactive Threshold Explorer for the MLP
#
# Move the slider to change the classification threshold applied to the MLP's
# predicted probabilities, and watch accuracy, precision, recall, F1-score,
# and the confusion matrix update immediately.

def explore_threshold(threshold=0.50):
    y_pred_t = (mlp_test_probs >= threshold).astype(int)

    acc = accuracy_score(y_test, y_pred_t)
    prec = precision_score(y_test, y_pred_t, zero_division=0)
    rec = recall_score(y_test, y_pred_t, zero_division=0)
    f1 = f1_score(y_test, y_pred_t, zero_division=0)
    cm = confusion_matrix(y_test, y_pred_t)

    print(f"Threshold : {threshold:.2f}")
    print(f"Accuracy  : {acc:.4f}")
    print(f"Precision : {prec:.4f}")
    print(f"Recall    : {rec:.4f}")
    print(f"F1-score  : {f1:.4f}")

    fig = go.Figure(data=go.Heatmap(
        z=cm, x=labels, y=labels, colorscale="Blues",
        text=cm, texttemplate="%{text}", showscale=False
    ))
    fig.update_xaxes(title_text="Predicted Label")
    fig.update_yaxes(title_text="True Label", autorange="reversed")
    fig.update_layout(
        title=f"MLP Confusion Matrix at Threshold = {threshold:.2f}",
        height=420, width=420, template="plotly_white"
    )
    fig.show()

interact(
    explore_threshold,
    threshold=FloatSlider(value=0.50, min=0.05, max=0.95, step=0.05,
                           description="Threshold", continuous_update=False)
);

interactive(children=(FloatSlider(value=0.5, continuous_update=False, description='Threshold', max=0.95, min=0…

```markdown
## Multiclass Experiment on the Iris Dataset

We will now replicate the entire workflow using the **Iris dataset**. Since this is a 3-class classification task, we will use a `softmax` activation function in the output layer of both our Perceptron and Multilayer Perceptron models and optimize them with `sparse_categorical_crossentropy` loss.
```

In [ ]:
from sklearn.datasets import load_iris

# Load the Iris dataset
iris = load_iris(as_frame=True)
iris_df = iris.frame
display(iris_df.head())

X_iris = iris.data
y_iris = iris.target

print("Shape of features:", X_iris.shape)
print("Class distribution:\n", y_iris.value_counts().sort_index())
print("Target names:", iris.target_names)

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


Shape of features: (150, 4)
Class distribution:
 target
0    50
1    50
2    50
Name: count, dtype: int64
Target names: ['setosa' 'versicolor' 'virginica']


In [ ]:
# Split and Standardize
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_iris_train_full, X_iris_test, y_iris_train_full, y_iris_test = train_test_split(
    X_iris, y_iris,
    test_size=0.2,
    random_state=42,
    stratify=y_iris
)

X_iris_train, X_iris_val, y_iris_train, y_iris_val = train_test_split(
    X_iris_train_full, y_iris_train_full,
    test_size=0.2,
    random_state=42,
    stratify=y_iris_train_full
)

iris_scaler = StandardScaler()
X_iris_train_scaled = iris_scaler.fit_transform(X_iris_train)
X_iris_val_scaled = iris_scaler.transform(X_iris_val)
X_iris_test_scaled = iris_scaler.transform(X_iris_test)

print("Iris Train shape:", X_iris_train_scaled.shape)
print("Iris Validation shape:", X_iris_val_scaled.shape)
print("Iris Test shape:", X_iris_test_scaled.shape)

Iris Train shape: (96, 4)
Iris Validation shape: (24, 4)
Iris Test shape: (30, 4)


In [ ]:
import tensorflow as tf

# 1. Single-Layer Perceptron (Multiclass Softmax Regression)
iris_perceptron = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_iris_train_scaled.shape[1],)),
    tf.keras.layers.Dense(3, activation="softmax"),
])

iris_perceptron.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# 2. Multilayer Perceptron (MLP with Softmax Output)
iris_mlp = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_iris_train_scaled.shape[1],)),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(3, activation="softmax")
])

iris_mlp.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

iris_perceptron.summary()
iris_mlp.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_8 (Dense)                 │ (None, 3)              │            15 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15 (60.00 B)

 Trainable params: 15 (60.00 B)

 Non-trainable params: 0 (0.00 B)

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_9 (Dense)                 │ (None, 16)             │            80 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 3)              │            27 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 243 (972.00 B)

 Trainable params: 243 (972.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
import tensorflow as tf

# Train Perceptron
iris_perceptron_history = iris_perceptron.fit(
    X_iris_train_scaled, y_iris_train,
    validation_data=(X_iris_val_scaled, y_iris_val),
    epochs=100,
    batch_size=16,
    verbose=0,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)]
)

# Train MLP
iris_mlp_history = iris_mlp.fit(
    X_iris_train_scaled, y_iris_train,
    validation_data=(X_iris_val_scaled, y_iris_val),
    epochs=100,
    batch_size=16,
    verbose=0,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)]
)

print(f"Iris Perceptron trained for {len(iris_perceptron_history.history['loss'])} epochs.")
print(f"Iris MLP trained for {len(iris_mlp_history.history['loss'])} epochs.")

Iris Perceptron trained for 100 epochs.
Iris MLP trained for 52 epochs.


In [ ]:
import numpy as np
from sklearn.metrics import classification_report

# Generate predictions (probabilities and hard class labels)
iris_perceptron_probs = iris_perceptron.predict(X_iris_test_scaled, verbose=0)
iris_perceptron_pred = np.argmax(iris_perceptron_probs, axis=1)

iris_mlp_probs = iris_mlp.predict(X_iris_test_scaled, verbose=0)
iris_mlp_pred = np.argmax(iris_mlp_probs, axis=1)

print("=== Perceptron Classification Report ===")
print(classification_report(y_iris_test, iris_perceptron_pred, target_names=iris.target_names))

print("=== MLP Classification Report ===")
print(classification_report(y_iris_test, iris_mlp_pred, target_names=iris.target_names))

=== Perceptron Classification Report ===
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      0.90      0.95        10
   virginica       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30

=== MLP Classification Report ===
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      0.90      0.95        10
   virginica       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30



In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Interactive learning curves comparison for the Iris experiment
fig = make_subplots(rows=1, cols=2, subplot_titles=("Perceptron Iris Loss", "MLP Iris Loss"))

fig.add_trace(go.Scatter(y=iris_perceptron_history.history["loss"], mode="lines", name="Train",
                          line=dict(color="#2a9d8f")), row=1, col=1)
fig.add_trace(go.Scatter(y=iris_perceptron_history.history["val_loss"], mode="lines", name="Validation",
                          line=dict(color="#e76f51")), row=1, col=1)

fig.add_trace(go.Scatter(y=iris_mlp_history.history["loss"], mode="lines", name="Train",
                          line=dict(color="#2a9d8f"), showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(y=iris_mlp_history.history["val_loss"], mode="lines", name="Validation",
                          line=dict(color="#e76f51"), showlegend=False), row=1, col=2)

fig.update_xaxes(title_text="Epoch")
fig.update_yaxes(title_text="Categorical Cross-Entropy Loss")
fig.update_layout(title="Iris Experiment: Training vs Validation Loss", height=450, width=950,
                   template="plotly_white")
fig.show()

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import confusion_matrix

# Confusion Matrix comparison for Iris dataset
iris_p_cm = confusion_matrix(y_iris_test, iris_perceptron_pred)
iris_m_cm = confusion_matrix(y_iris_test, iris_mlp_pred)
labels = list(iris.target_names)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Perceptron Iris", "MLP Iris"))

fig.add_trace(go.Heatmap(z=iris_p_cm, x=labels, y=labels, colorscale="YlGnBu",
                          text=iris_p_cm, texttemplate="%{text}", showscale=False), row=1, col=1)
fig.add_trace(go.Heatmap(z=iris_m_cm, x=labels, y=labels, colorscale="YlGnBu",
                          text=iris_m_cm, texttemplate="%{text}", showscale=False), row=1, col=2)

fig.update_xaxes(title_text="Predicted Label")
fig.update_yaxes(title_text="True Label", autorange="reversed")
fig.update_layout(title="Iris Confusion Matrix Comparison", height=450, width=950,
                   template="plotly_white")
fig.show()